# Creates combined LULUCF tables (long and wide format)

Table manipulation from Claude session 'LULUCF and component table creation'

This involves giant tables, so some steps take many minutes. The longest step is converting the long-format veg tile-level tables to wide-format (>1 hr). 
Also, because this involves giant tables, it uses a lot of memory and doesn't seem to be able to run beginning to end on my computer; the kernel keeps crashing at one point or another. 
But by running a few steps, saving intermediate outputs to parquet, letting the kernel crash when it wants and then continuing by reading the parquets I saved, it's gotten all the way through.  

In [1]:
import pandas as pd
import yaml
import geodatasets
import geopandas as gpd
import math
import rasterio
import os
import openpyxl
import pygwalker as pyg
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import re
import glob
import gc
import sys
import textwrap
from shapely.geometry import Point
from datetime import datetime
from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from matplotlib.transforms import blended_transform_factory
from matplotlib.gridspec import GridSpec
from io import BytesIO
from IPython.display import Image, display as ipy_display
from rasterio.windows import Window
from tqdm import tqdm

In [2]:
# To add project files
# Keeps going up project structure until it gets to the root.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69a5ff07-91e0-8329-88d1-cf2ea6c159c2
project_root = Path().resolve()
while project_root.name != "AFOLU_GHG_flux_model":
    project_root = project_root.parent

sys.path.append(str(project_root))

from src.utilities import constants_and_names as cn
from src.utilities import universal_utilities as uu

now = datetime.now()
today = now.strftime("%Y%m%d")

pd.set_option('display.max_columns', None)

### Input table paths (zonal stats, and chunk stats for comparison purposes)

In [3]:
# Vegetation zonal stats output folder (separate parquet for each tile-- too large to combine into a single long-format parquet)
veg_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/vegetation_v{cn.veg_model_version_underscore}_standard_global__full_run_with_Canada_rerun__20260603/'

veg_wide_folder = f'{veg_zonal_stats_folder}wide_tiles/'
os.makedirs(veg_wide_folder, exist_ok=True)

In [4]:
veg_parquet_files = sorted(glob.glob(f'{veg_zonal_stats_folder}*.parquet'))
# veg_parquet_files_subset = veg_parquet_files[0:5]
print(f"Found {len(veg_parquet_files)} parquet files")

Found 357 parquet files


In [5]:
# SOC (including mineral soil) zonal stats output
SOC_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/SOC_v{cn.SOC_model_version_underscore}_standard_global__20260611/'
# SOC_parquet_name = f'SOC_model_zonal_stats_v{cn.SOC_model_version_underscore}_20260615_22_25_53.parquet'
SOC_wide_folder = f'{SOC_zonal_stats_folder}wide_tiles/'
os.makedirs(SOC_wide_folder, exist_ok=True)

In [6]:
SOC_parquet_files = sorted(glob.glob(f'{SOC_zonal_stats_folder}*.parquet'))
# SOC_parquet_files_subset = SOC_parquet_files[0:5]
print(f"Found {len(SOC_parquet_files)} parquet files")

Found 277 parquet files


In [7]:
# Organic soil zonal stats output
org_soil_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/organic_soil_v1_0_1_standard_global__20260610/'
org_soil_parquet_name = 'master_zonal_full_disaggregation.parquet'  

In [8]:
# LULUCF outputs
LULUCF_zonal_stats_folder = Path(f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/LULUCF_v{cn.LULUCF_model_version_underscore}__veg_v{cn.veg_model_version_underscore}__minsoil_v{cn.SOC_model_version_underscore}__orgsoil_v{cn.organic_soil_model_version_underscore}/')
LULUCF_zonal_stats_folder.mkdir(parents=True, exist_ok=True)
LULUCF_zonal_stats_folder

PosixPath('/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/LULUCF_v1_0_0__veg_v1_0_5__minsoil_v1_0_1__orgsoil_v1_0_1')

In [ ]:
# Contextual columns for making a unified wide table format

agg_layers = ['analysis_layer', 'year', 
              'land_state_node', 'land_state_meaning', 'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type',
              'adm0', 'country_name', 'region_L1', 'region_L2_L3',
              'continent', 'ecozone', 'continent_ecozone', 'climate_domain', 'cont_eco',
              'Landmark',
              'starting_composite_primary_forest',
              'KBA',
              'watershed', 'watershed_name',
              'drivers_of_TCL_1_km', 'driver_1km_text',
              'forest_age_category_end_of_interval',
              # 'gas',
              'WDPA', 'WDPA_type', 'WDPA_high_protection',
              'first_year_LC_composite_name'  # SOC only
             ]

In [ ]:
# Converts long-format parquet tables for individual tiles to wide format. Used for vegetation and SOC.
def convert_indiv_tiles_to_wide(tile_parquets, context_layers, out_folder, LULUCF_component, model_version, context_layers_description):

    print("Converting long tables to wide")
    
    os.makedirs(out_folder, exist_ok=True)
    
    # Contextual columns
    contextual_cols = [c for c in context_layers if c != 'analysis_layer']

    wide_columns = []
    row_count = 0

    # Iterates through tables
    for f in tqdm(tile_parquets):
        tile_id = os.path.splitext(os.path.basename(f))[0]   # e.g. "00N_020E"
        out_path = f'{out_folder}{tile_id}_v{model_version}__{context_layers_description}__wide__{today}.parquet'

        # Counts rows in any wide tables that have already been created
        # Rather than reading in the already existing files, it reads just their metadata
        if os.path.exists(out_path):
            pf_done = pq.ParquetFile(out_path)
            wide_columns = pq.read_schema(out_path).names
            row_count += pf_done.metadata.num_rows
            continue

        # Read only columns that exist in this tile
        available_cols = set(pq.read_schema(f).names)
        cols_to_read   = [c for c in agg_layers + ['value', 'area_ha'] if c in available_cols]
        missing_cols   = [c for c in agg_layers if c not in available_cols]

        year_chunk_tile = pd.read_parquet(f, columns=cols_to_read)
        df_tile = df_tile.drop(columns=['gas'], errors='ignore')

        for col in missing_cols:
            df_tile[col] = 'Unassigned'
        
        # Aggregates wide table by selected contextual layers
        agg = (
            df_tile
            .groupby(agg_layers, dropna=False)[['value', 'area_ha']]
            .sum()
            .reset_index()
        )

        # Separate processing of fluxes and areas
        flux_wide = (
            agg.set_index(contextual_cols + ['analysis_layer'])['value']
            .unstack('analysis_layer')
            .rename_axis(None, axis='columns')
            .fillna(0)
            .reset_index()
        )
        area_wide = (
            agg.set_index(contextual_cols + ['analysis_layer'])['area_ha']
            .unstack('analysis_layer')
            .add_suffix('__area_ha')
            .rename_axis(None, axis='columns')
            .fillna(0)
            .reset_index()
        )

        # Adds "veg" to front of analysis column names
        flux_wide = flux_wide.rename(columns={
            col: f'{LULUCF_component}{col}'
            for col in flux_wide.columns if col not in contextual_cols
        })

        # Strips _ha from flux and area columns to clarify that these are stocks and not densities
        flux_wide.columns = [
            re.sub(r'_MgC_ha$', '_MgC', col)
            for col in flux_wide.columns
        ]
        area_wide.columns = [
            re.sub(r'_MgC_ha$', '_MgC', col)
            for col in area_wide.columns
        ]

        # Replaces density with stock because these are stocks over an area
        flux_wide.columns = [
            re.sub(r'_density_', '_stock_', col)
            for col in flux_wide.columns
        ]
        area_wide.columns = [
            re.sub(r'_density_', '_stock_', col)
            for col in area_wide.columns
        ]

        # Specifies that flux columns are annual values by adding _yr
        flux_wide.columns = [
            re.sub(r'MgCO2e?', r'\g<0>_yr', col)
            for col in flux_wide.columns
        ]
        
        # Strip flux units from area columns: e.g. gross_emissions__AGC__MgCO2__area_ha → gross_emissions__AGC__area_ha
        area_wide.columns = [
            re.sub(r'_+(MgCO2e|MgCO2|MgC_ha|MgC|fraction)(__area_ha)$', r'\2', col)
            for col in area_wide.columns
        ]
        area_wide = area_wide.rename(columns={
            col: f'{LULUCF_component}{col}'
            for col in area_wide.columns if col not in contextual_cols
        })

        # Changes the way the soil depth is specified
        flux_wide.columns = [
            re.sub(r'0-30cm', '0_30cm', col)
            for col in flux_wide.columns
        ]
        area_wide.columns = [
            re.sub(r'0-30cm', '0_30cm', col)
            for col in area_wide.columns
        ]

        # Merges area and flux tables back together
        tile_wide = flux_wide.merge(area_wide, on=contextual_cols)
        
        tile_wide.to_parquet(out_path, index=False)
        # display(tile_wide)

        wide_columns = tile_wide.columns.to_list()
        row_count += len(tile_wide)

        # Saves first wide tile as csv for inspection
        if "00N_000E" in tile_id:
            out_path_csv = out_path.replace('.parquet', '.csv')
            tile_wide.to_csv(out_path_csv, index=False)

    print(f"Saved {len(tile_parquets)} wide tile parquets to {out_folder}")

    return wide_columns, row_count, contextual_cols

In [ ]:
# Combines wide-format tables for individual tiles into global composite, including reaggregating across tiles (since no tile_id field anymore). Used for vegetation and SOC.
def combine_wide_files(wide_folder, main_folder, contextual_cols, out_name, version, context_layers_description):

    wide_tile_files = sorted(glob.glob(f'{wide_folder}/*wide*.parquet'))
    print(f"Found {len(wide_tile_files)} wide parquet files to combine")

    # Combines all tile tables
    print("Combining tile-level wide parquets")
    combined = pd.concat(
        [pd.read_parquet(f) for f in tqdm(wide_tile_files)],
        ignore_index=True
    )

    # Reaggregates with remaining contextual layers
    print("Reaggregating global wide parquet")
    sums_wide = (
        combined
        .groupby(contextual_cols, dropna=False)
        .sum(numeric_only=True)
        .reset_index()
    )

    sums_wide.to_parquet(f"{main_folder}/{out_name}__global__v{version}__{context_layers_description}__wide__{today}.parquet", index=False)

    return sums_wide

### Global vegetation table with full contextual layers

In [ ]:
%%time

# Converts all long parquet tables to wide
# Took 1h 5min with all contextual layers

veg_wide_columns, veg_row_count, veg_contextual_cols = convert_indiv_tiles_to_wide(veg_parquet_files, agg_layers, veg_wide_folder, "veg_", cn.veg_model_version_underscore, "all_vars")
print(f"Total rows across wide tiles: {veg_row_count:}")
print(f"Contextual columns across wide tiles: {veg_contextual_cols}")
print(f"Wide columns: {veg_wide_columns}")

In [ ]:
%%time

# Combines all tile-level wide tables into a single global wide table and reaggregates with contextual layers 
# (i.e. sums rows that have duplicate contextual combinations across tiles because there's no tile_id)

veg_sums_wide = combine_wide_files(veg_wide_folder, veg_zonal_stats_folder, veg_contextual_cols, "veg_model_zonal_stats", cn.veg_model_version_underscore, "all_vars")
print(len(veg_sums_wide))
display(veg_sums_wide)

### Standardizing soil tables

In [ ]:
# Fills in intervening years for organic soil data.
# For organic soil, 2020 interval is applied to 2016-2020 and 2024 is applied to 2021-2024. 
# Per Claude session 'LULUCF and component table creation'
def fill_in_soil_years(df, group_cols, agg_first=False):
    """
    Expands multi-year SOC intervals into individual annual rows, then extends
    through the end of the vegetation timeseries (cn.years_annual[-1]).

    soil_type='organic':
        Keeps 5-year endpoints AND the final available year even if not divisible
        by 5 (e.g. 2022). 5-year endpoints each cover 5 years; the final non-5
        year covers from the year after the previous 5-year endpoint through
        itself. The final year is extended forward through cn.years_annual[-1]
        if needed. Result: 2016-2020 carry the 2020 value, 2021-2024 carry the
        2022 (or later) value.
    """

    # Reaggregates by all remaining contextual columns
    if agg_first:
        df = (
            df
            .groupby(group_cols, dropna=False, as_index=False)
            .agg({
                "zonal_stats_sum": "sum",
                "area_ha": "sum"
            })
        )

    df = df.copy()
    df["year"] = df["year"].astype(int)

    # Keep 5-year endpoints and the final available year
    last_year_in_data = int(df['year'].max())
    df = df[df['year'].mod(5).eq(0) | df['year'].eq(last_year_in_data)].copy()

    if df.empty:
        return df

    df = df.sort_values(group_cols + ["year"]).reset_index(drop=True)

    year_vals = df["year"].astype(int)
    last_year = int(year_vals.max())

    # Compute how many years each row covers:
    #   - 5-year endpoint: always covers 5 years (the 5 years ending at that year)
    #   - non-mod-5 final year (organic only): covers years since the prior 5-year endpoint
    interval_len = np.where(
        year_vals.mod(5).eq(0),
        5,
        last_year - (last_year // 5) * 5   # only reached for organic's non-mod-5 final year
    )

    start_year = (year_vals.to_numpy() - interval_len + 1).astype(int)
    end_year   = year_vals.to_numpy().astype(int)
    n_rep      = end_year - start_year + 1

    expanded = df.loc[df.index.repeat(n_rep)].copy()
    expanded["year"] = np.concatenate([
        np.arange(s, e + 1) for s, e in zip(start_year, end_year)
    ])
    expanded = expanded.reset_index(drop=True)

    # Extend the last available year forward through the vegetation timeseries end
    if last_year < cn.years_annual[-1]:
        rows_to_extend = expanded[expanded["year"] == last_year].copy()
        extra = pd.concat(
            [rows_to_extend.assign(year=yr) for yr in range(last_year + 1, cn.years_annual[-1] + 1)],
            ignore_index=True
        )
        expanded = pd.concat([expanded, extra], ignore_index=True)
        expanded = expanded.sort_values(group_cols + ["year"]).reset_index(drop=True)

    return expanded

In [ ]:
%%time
### Standardizes SOC table 

# Converts each tile's parquet from long to wide
wide_columns, row_count, contextual_cols = convert_indiv_tiles_to_wide(SOC_parquet_files, agg_layers, SOC_wide_folder, "", cn.SOC_model_version_underscore, "all_vars")
print(f"Total rows across wide tiles: {row_count:}")
print(f"Contextual columns across wide tiles: {contextual_cols}")
print(f"Wide columns: {wide_columns}")

# Combines all wide tables into a single table
SOC_sums_wide = combine_wide_files(SOC_wide_folder, SOC_zonal_stats_folder, contextual_cols, "SOC_model_zonal_stats", cn.SOC_model_version_underscore, "all_vars_all_years")
print(len(SOC_sums_wide))
display(SOC_sums_wide)

# Drops all years before vegetation data (before 2016) because we don't need those for LULUCF (i.e. keeps only the 2020 and 2022 stock and change data)
SOC_2020_wide = SOC_sums_wide[SOC_sums_wide["year"] == 2020].reset_index(drop=True)
print(f"Rows in SOC_years_filled_wide: {len(SOC_2020_wide)}")
display(SOC_2020_wide)

SOC_years_filled_wide = pd.concat(
    [SOC_2020_wide.assign(year=yr) for yr in cn.interval_end_years_annual],
    ignore_index=True
)
display(SOC_years_filled_wide)

# Information on the years included in the table
SOC_year_count = len(SOC_years_filled_wide['year'].unique())
print(f"Years in SOC data: {SOC_years_filled_wide['year'].unique()} is {SOC_year_count} years")
print(SOC_years_filled_wide.groupby('year').size())


# Adds contextual columns found in the vegetation table. Fills with Unassigned for text fields and -999 for numeric fields.
# Gets the data type for the fields from the vegetation table
veg_parquet_path = f'{veg_zonal_stats_folder}veg_model_zonal_stats__global__v{cn.veg_model_version_underscore}__all_vars__wide__20260616.parquet'
veg_field_types = {field.name: field.type for field in pq.read_schema(veg_parquet_path)}
contextual_cols = [c for c in agg_layers if c != 'analysis_layer']
cols_added = []
for col in contextual_cols:
    if col not in SOC_years_filled_wide.columns:
        arrow_type = veg_field_types.get(col)
        is_numeric = arrow_type is not None and (pa.types.is_integer(arrow_type) or pa.types.is_floating(arrow_type))
        default = -999 if is_numeric else 'Unassigned'
        SOC_years_filled_wide[col] = default
        print(f"  Adding {col} with default of {default}")
        cols_added.append(col)
print(f"Done adding columns. Added {len(cols_added)} columns ({cols_added}).")

# QC: Timeseries of SOC change (full and mineral extent) to compare against chunk stats (Mg CO2/yr)
# Some chunk stat values to check against: Density mineral extent 2020 = 350361333214; Net mineral extent 2020=1264590854; gain mineral extent 2020= -5495786224; loss mineral extent 2020= 6760376953
print(SOC_years_filled_wide['SOC_stock__mineral_soil_extent__0_30cm_MgC'].sum()/9)
print(SOC_years_filled_wide['SOC_net__mineral_soil_extent__0_30cm_MgCO2_yr'].sum()/9)
print(SOC_years_filled_wide['SOC_gain__mineral_soil_extent__0_30cm_MgCO2_yr'].sum()/9)
print(SOC_years_filled_wide['SOC_loss__mineral_soil_extent__0_30cm_MgCO2_yr'].sum()/9)


SOC_years_filled_wide.to_parquet(f'{SOC_zonal_stats_folder}SOC_model_zonal_stats__global__v{cn.SOC_model_version_underscore}__post_2016__all_vars__wide__{today}.parquet', index=False)
# SOC_years_filled_wide = pd.read_parquet(f'{SOC_zonal_stats_folder}SOC_model_zonal_stats__global__v{cn.SOC_model_version_underscore}__post_2016__all_vars__wide__20260616.parquet')
SOC_years_filled_wide

In [ ]:
### Global SOC gross loss and gain by interval: full extent only (gain positive, loss negative, counter to LULUCF convention)
### Per Claude session 'Global SOC loss/gain by year'

# ── Data ─────────────────────────────────────────────────────────────────────
FULL_LOSS = 'SOC_loss__full_extent__0-30cm_MgCO2'
FULL_GAIN = 'SOC_gain__full_extent__0-30cm_MgCO2'
FULL_NET  = 'SOC_net__full_extent__0-30cm_MgCO2'

df_soc = SOC_df_raw[SOC_df_raw['analysis_layer'].isin([FULL_LOSS, FULL_GAIN, FULL_NET])].copy()

year_totals = (
    df_soc
    .groupby(['year', 'analysis_layer'])['zonal_stats_sum']
    .sum()
    .div(1e9)
    .unstack('analysis_layer')
    .rename_axis(None, axis=1)
    .sort_index()
)

years = year_totals.index.tolist()
x = np.arange(len(years))

# Flip signs: gain → positive, loss → negative, net → flipped accordingly
loss_vals = -year_totals[FULL_LOSS].values   # was positive, now negative
gain_vals = -year_totals[FULL_GAIN].values   # was negative, now positive
net_vals  = -year_totals[FULL_NET].values

# Scale final interval: ×2 ÷ 3.5
# In the original geoprocessing, I incorrectly divided the 2022 change block by 2 to annualize instead of dividing by 3.5. This correct that. 
loss_vals[-1] = loss_vals[-1] * 2 / 3.5
gain_vals[-1] = gain_vals[-1] * 2 / 3.5
net_vals[-1]  = net_vals[-1]  * 2 / 3.5

# ── Colors (distinct for this figure) ────────────────────────────────────────
C_LOSS = "#8b008b"   # dark magenta — loss (emission)
C_GAIN = "#33cc33"   # bright green — gain (removal)
C_NET  = "#000000"   # black   — net

# ── Plot ──────────────────────────────────────────────────────────────────────
bar_w = 0.6

fig, ax = plt.subplots(figsize=(7, 5))

ax.bar(x, loss_vals, width=bar_w, color=C_LOSS, zorder=2, label='Loss')
ax.bar(x, gain_vals, width=bar_w, color=C_GAIN, zorder=2, label='Gain')
ax.plot(x, net_vals, color=C_NET, marker='o', linewidth=1.8, zorder=4, label='Net change')

ax.axhline(0, color='black', linewidth=0.8, zorder=1)
ax.yaxis.grid(True, color='lightgrey', linewidth=0.8)
ax.set_axisbelow(True)

ax.set_xticks(x)
ax.set_xticklabels(years)
ax.set_ylabel('Gt CO₂/yr')
ax.set_title('Global SOC loss and gain, full extent, 0–30 cm, \n3.5 year denomination in final interval', fontsize=11)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

legend_handles = [
    mpatches.Patch(color=C_GAIN, label='Gain (positive = removal)'),
    mpatches.Patch(color=C_LOSS, label='Loss (negative = emission)'),
    plt.Line2D([0], [0], color=C_NET, marker='o', linewidth=1.8, label='Net change'),
]
ax.legend(handles=legend_handles, frameon=False, fontsize=9)

plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/SOC_loss_gain_full_extent_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

year_totals_display = year_totals.copy()
year_totals_display.iloc[-1] *= 2 / 3.5   # annualise final interval to match chart
display(year_totals_display[[FULL_LOSS, FULL_GAIN, FULL_NET]].round(3))


# ── Country-level ratio table: annualized 2022 vs annualized 2020 ─────────────
# To understand in which countries the jump in gross loss and gain in the last change interval is occurring 
COUNTRY_COL   = 'adm0'   

df_ctry = (
    SOC_df_raw[
        SOC_df_raw['analysis_layer'].isin([FULL_LOSS, FULL_GAIN]) &
        SOC_df_raw['year'].isin([2020, 2022])
    ]
    .groupby([COUNTRY_COL, 'year', 'analysis_layer'])['zonal_stats_sum']
    .sum()
    .unstack(['year', 'analysis_layer'])
)
df_ctry.columns = [f'{yr}_{lyr}' for yr, lyr in df_ctry.columns]

# Annualise: 2022 → ×2÷3.5 (matches chart scaling-- which has incorrectly divided the 2022 change block by 2 to annualize instead of 3.5); 2020 → is already annualized
ann_gain_2020 = df_ctry[f'2020_{FULL_GAIN}'] 
ann_gain_2022 = df_ctry[f'2022_{FULL_GAIN}'] * 2 / 3.5
ann_loss_2020 = df_ctry[f'2020_{FULL_LOSS}'] 
ann_loss_2022 = df_ctry[f'2022_{FULL_LOSS}'] * 2 / 3.5

ratio_table = pd.DataFrame({
    'Ann. gain 2020 (Gt/yr)': ann_gain_2020 / 1e9,
    'Ann. gain 2022 (Gt/yr)': ann_gain_2022 / 1e9,
    'Gain ratio (2022/2020)':  ann_gain_2022 / ann_gain_2020,
    'Ann. loss 2020 (Gt/yr)': ann_loss_2020 / 1e9,
    'Ann. loss 2022 (Gt/yr)': ann_loss_2022 / 1e9,
    'Loss ratio (2022/2020)':  ann_loss_2022 / ann_loss_2020,
}).sort_values('Ann. gain 2020 (Gt/yr)', ascending=False)

display(ratio_table.round(3).head(10))

In [ ]:
%%time
### Standardizes organic soil table with vegetation table (already in wide format)

# Reads organic soil zonal stats parquet table
org_soil_raw_wide = pd.read_parquet(f'{org_soil_zonal_stats_folder}{org_soil_parquet_name}')
org_soil_raw_wide
print(f"Rows in org_soil_raw_wide: {len(org_soil_raw_wide)}")

# Makes table columns generally match vegetation and SOC
org_soil_adjusted_wide = org_soil_raw_wide.drop(columns=['gadm_adm0', 'country', 'inventory_period', 'interval_start', 'land_use', 'drainage_class', 
                                                         'drained_state_meaning', 'burned_state_meaning', 'combined_state_nodes',
                                                         'drained_state_nodes', 'burned_state_nodes'])
display(org_soil_adjusted_wide.columns)

# Renames columns to match vegetation names
org_soil_adjusted_wide.rename(columns={'iso3': 'adm0', 
                                'component': 'analysis_layer',
                                'interval_end': 'year',
                                'river_basins': 'watershed',
                                'wdpa': 'WDPA',
                                'kba': 'KBA',
                                'landmark': 'Landmark',
                                'area__ha': 'org_soil__area_ha'},
                       inplace=True)

# Fills NaN (empty) values with 0s
org_soil_adjusted_wide = org_soil_adjusted_wide.fillna(0)
# org_soil_adjusted_wide

# Renames columns
org_soil_adjusted_wide.columns = org_soil_adjusted_wide.columns.str.replace('_Mg_CO2', '__MgCO2')
org_soil_adjusted_wide.columns = [re.sub(r'MgCO2e?', r'\g<0>_yr', col) for col in org_soil_adjusted_wide.columns]
org_soil_adjusted_wide.columns = org_soil_adjusted_wide.columns.str.replace('drained', 'org_soil_drained')
org_soil_adjusted_wide.columns = org_soil_adjusted_wide.columns.str.replace('burned', 'org_soil_burned')
org_soil_adjusted_wide.columns = org_soil_adjusted_wide.columns.str.replace('co2', 'CO2')
org_soil_adjusted_wide.columns = org_soil_adjusted_wide.columns.str.replace('ch4', 'CH4')
org_soil_adjusted_wide.columns = org_soil_adjusted_wide.columns.str.replace('n2o', 'N2O')

# Adds attribute columns that are derived from existing columns. This is done in create_df() in zonal_stats_utilities.py for the vegetation and SOC data. 
org_soil_adjusted_wide['country_name'] = org_soil_adjusted_wide[cn.adm0_pattern].map(cn.iso_to_country)
org_soil_adjusted_wide['region_L1'] = org_soil_adjusted_wide[cn.adm0_pattern].map(cn.iso_to_region_UN_geoscheme_L1)
org_soil_adjusted_wide['region_L2_L3'] = org_soil_adjusted_wide[cn.adm0_pattern].map(cn.iso_to_region_UN_geoscheme_L2_L3)

# Because some rows for contextual layers may be blank
org_soil_adjusted_wide[cn.adm0_pattern] = org_soil_adjusted_wide[cn.adm0_pattern].fillna("Unassigned")
org_soil_adjusted_wide['country_name'] = org_soil_adjusted_wide['country_name'].fillna("Unassigned")
org_soil_adjusted_wide['region_L1'] = org_soil_adjusted_wide['region_L1'].fillna("Unassigned")
org_soil_adjusted_wide['region_L2_L3'] = org_soil_adjusted_wide['region_L2_L3'].fillna("Unassigned")

org_soil_adjusted_wide["climate_domain"] = org_soil_adjusted_wide["climate_domain"].replace({         
    "tropical": "Subtropical/tropical",
    "temperate": "Temperate",
    "boreal": "Boreal",
    "Unspecified": "Unassigned",
    "other_domain": "Unassigned"
})

org_soil_adjusted_wide['watershed_name'] = org_soil_adjusted_wide[cn.watersheds_pattern].map(cn.watershed_to_text)
org_soil_adjusted_wide["watershed_name"] = org_soil_adjusted_wide["watershed_name"].fillna("Unassigned")

org_soil_adjusted_wide['WDPA_type'] = org_soil_adjusted_wide[cn.WDPA_pattern].map(cn.WDPA_to_text)

# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69aee45e-ce6c-8325-b1d0-a6c6b0e7ae2e
org_soil_adjusted_wide["WDPA_high_protection"] = "Other protection status"

org_soil_adjusted_wide.loc[org_soil_adjusted_wide["WDPA_type"] == "NA", "WDPA_high_protection"] = "Not protected"
org_soil_adjusted_wide.loc[org_soil_adjusted_wide["WDPA_type"].isin(["Category Ia", "Category Ib", "Category II", "Category III"]), "WDPA_high_protection"] = "High protection"

org_soil_adjusted_wide['driver_1km_text'] = org_soil_adjusted_wide[cn.drivers_of_loss_pattern].map(cn.drivers_to_text)
org_soil_adjusted_wide['driver_1km_text'] = org_soil_adjusted_wide['driver_1km_text'].fillna("Unassigned")
# org_soil_adjusted_wide

# Reaggregates by remaining contextual columns after dropped columns are removed
analysis_cols = [col for col in org_soil_adjusted_wide.columns if 'MgCO2' in col] + ['org_soil__area_ha']
contextual_cols_adj = [col for col in org_soil_adjusted_wide.columns if col not in analysis_cols]
org_soil_adjusted_wide = (
    org_soil_adjusted_wide
    .groupby(contextual_cols_adj, dropna=False)[analysis_cols]
    .sum()
    .reset_index()
)
print(f"Rows in org_soil_adjusted_wide after reaggregation: {len(org_soil_adjusted_wide)}")

# Drops all years before vegetation data (before 2016) because we don't need those for LULUCF (i.e. keeps only the 2020 and 2022 stock and change data)
org_soil_post_2016_wide = org_soil_adjusted_wide[org_soil_adjusted_wide["year"] >= cn.interval_end_years_annual[0]].reset_index(drop=True)
print(f"Rows in org_soil_post_2016_wide: {len(org_soil_post_2016_wide)}")
org_soil_post_2016_wide


# Fills in the years for multi-year interval organic soil data.
# 2020 is copied to 2016-2020 and 2022 to 2021-2024 (to match end of vegetation timeseries). 
# If a year doesn't have data for a given year and combination of contextual layers, there is nothing to expand and all years in that interval are empty.
analysis_cols = [col for col in org_soil_post_2016_wide.columns if 'Mg_CO2' in col] + ['org_soil__area_ha'] # Identifies non-contextual columns
contextual_cols = org_soil_post_2016_wide.drop(columns=analysis_cols).columns.to_list()
# display(series_cols)
org_soil_post_2016_filled_in_wide = fill_in_soil_years(org_soil_post_2016_wide, contextual_cols, agg_first=False)
print(f"Rows in org_soil_post_2016_filled_in_wide: {len(org_soil_post_2016_filled_in_wide)}")

# Ratio below Should be very close to 4.5. If every contextual combination has every year, full year expansion would be 4.5 (2020 interval expanded 5x years, 2022 expanded 4x years).
# However, some contextual combinations don't have all years (usually because they're so rare and other years have just 1 pixel of that combination),
# so that doesn't get expanded to other years. This results in deviation from the 4.25x expansion. 
print(f"Ratio of rows in unexpanded to expanded tables: {len(org_soil_post_2016_filled_in_wide)/len(org_soil_post_2016_wide)}")

# Adds contextual columns found in the vegetation table. Fills with Unassigned for text fields and -999 for numeric fields.
# Gets the data type for the fields from the vegetation table
veg_parquet_path = f'{veg_zonal_stats_folder}veg_model_zonal_stats__global__v{cn.veg_model_version_underscore}__all_layers__wide__20260616.parquet'
veg_field_types = {field.name: field.type for field in pq.read_schema(veg_parquet_path)}
contextual_cols = [c for c in agg_layers if c != 'analysis_layer']
cols_added = []
for col in contextual_cols:
    if col not in org_soil_post_2016_filled_in_wide.columns:
        arrow_type = veg_field_types.get(col)
        is_numeric = arrow_type is not None and (pa.types.is_integer(arrow_type) or pa.types.is_floating(arrow_type))
        default = -999 if is_numeric else 'Unassigned'
        org_soil_post_2016_filled_in_wide[col] = default
        print(f"  Adding {col} with default of {default}")
        cols_added.append(col)
print(f"Done adding columns. Added {len(cols_added)} columns ({cols_added}).")

# Recasts float64 columns to float32 columns. Vegetation and SOC are already float32.
org_soil_post_2016_filled_in_wide[org_soil_post_2016_filled_in_wide.select_dtypes('float64').columns] = org_soil_post_2016_filled_in_wide.select_dtypes('float64').astype('float32')

# Information on the years included in the table
org_soil_year_count = len(org_soil_post_2016_filled_in_wide['year'].unique())
print(f"Years in organic soil data: {org_soil_post_2016_filled_in_wide['year'].unique()} is {org_soil_year_count} years")
print(org_soil_post_2016_filled_in_wide.groupby('year').size())

org_soil_post_2016_filled_in_wide.to_csv(f"{org_soil_zonal_stats_folder}org_soil__post_2016__all_vars__wide__{today}.csv", index=False)
org_soil_post_2016_filled_in_wide.to_parquet(f"{org_soil_zonal_stats_folder}org_soil__post_2016__all_vars__wide__{today}.parquet", index=False)
org_soil_post_2016_filled_in_wide

### Create combined LULUCF table (wide format)

In [10]:
# Read only the small tables into pandas
SOC_df_post_2016_years_filled_in_wide  = pd.read_parquet(f'{SOC_zonal_stats_folder}SOC_model_zonal_stats__global__v{cn.SOC_model_version_underscore}__post_2016__all_vars__wide__20260616.parquet')
org_soil_post_2016_filled_in_wide      = pd.read_parquet(f"{org_soil_zonal_stats_folder}org_soil__post_2016__all_vars__wide__20260616.parquet")

# veg stays on disk — reference only the path to keep it out of memory
veg_parquet_path = f'{veg_zonal_stats_folder}veg_model_zonal_stats__global__v{cn.veg_model_version_underscore}__all_layers__wide__20260616.parquet'
veg_schema = pq.read_schema(veg_parquet_path)
all_veg_cols = veg_schema.names
veg_metadata = pq.read_metadata(veg_parquet_path)
veg_row_count = veg_metadata.num_rows

SOC_table_shape = SOC_df_post_2016_years_filled_in_wide.shape
org_soil_table_shape = org_soil_post_2016_filled_in_wide.shape

print(f"Dimensions of SOC table:      {SOC_table_shape}")
print(f"Dimensions of org soil table: {org_soil_table_shape}")
print(f"Veg rows: {veg_row_count:,}")
print(f"Veg column names ({len(all_veg_cols)} columns): {all_veg_cols}")

Dimensions of SOC table:      (5788116, 43)
Dimensions of org soil table: (841518, 37)
Veg rows: 22,745,922
Veg column names (73 columns): ['year', 'land_state_node', 'land_state_meaning', 'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type', 'adm0', 'country_name', 'region_L1', 'region_L2_L3', 'continent', 'ecozone', 'continent_ecozone', 'climate_domain', 'cont_eco', 'Landmark', 'starting_composite_primary_forest', 'KBA', 'watershed', 'watershed_name', 'drivers_of_TCL_1_km', 'driver_1km_text', 'forest_age_category_end_of_interval', 'WDPA', 'WDPA_type', 'WDPA_high_protection', 'first_year_LC_composite_name', 'veg__AGC_emission_factor_CO2_only__fraction', 'veg__carbon_stock__non_soil__MgC', 'veg__gross_emissions__AGC__MgCO2_yr', 'veg__gross_emissions__BGC__MgCO2_yr', 'veg__gross_emissions__CH4__MgCO2e_yr', 'veg__gross_emissions__N2O__MgCO2e_yr', 'veg__gross_emissions__all_C_pools__CO2_only__MgCO2_yr', 'veg__gross_emissions__all_C_pools__all_gases__MgCO2e_yr', 'veg__gro

In [11]:
# Confirms that vegetation, SOC and organic soil tables have the right columns and the right options in each column
# Also, establishes contextual columns for each table

print(f"Columns in vegetation table are: {all_veg_cols}")
# Non-contexual column identifier strings
keep_strings = ['MgC', 'emission_factor', 'area']
veg_analysis_cols = [
    col for col in all_veg_cols
    if any(s in col for s in keep_strings)
]
# Contextual cols are everything that isn't a flux or area analysis column
veg_contextual_cols = [c for c in all_veg_cols if c not in veg_analysis_cols]
print(f"Contextual columns are: {veg_contextual_cols}")

print("\n")
print(f"Columns in mineral soil table are: {SOC_df_post_2016_years_filled_in_wide.columns}")
SOC_analysis_cols = [col for col in SOC_df_post_2016_years_filled_in_wide.columns if 'MgC' in col] # Identifies non-contextual columns
SOC_contextual_cols = SOC_df_post_2016_years_filled_in_wide.drop(columns=SOC_analysis_cols).columns.to_list()  # Identifies contextual col
print(f"Contextual columns are: {SOC_contextual_cols}")
# for column in SOC_contextual_cols:
#     print(f"{column}: {np.sort(SOC_df_post_2016_years_filled_in_wide[column].unique())}")

print("\n")
print(f"Columns in organic soil table are: {org_soil_post_2016_filled_in_wide.columns}")
org_soil_analysis_cols = [col for col in org_soil_post_2016_filled_in_wide.columns if 'MgC' in col] + ["org_soil__area_ha"]  # Identifies non-contextual columns
org_soil_contextual_cols = org_soil_post_2016_filled_in_wide.drop(columns=org_soil_analysis_cols).columns.to_list()  # Identifies contextual col
print(f"Contextual columns are: {org_soil_contextual_cols}")
# for column in org_soil_contextual_cols:
#     print(f"{column}: {np.sort(org_soil_post_2016_filled_in_wide[column].unique())}")

Columns in vegetation table are: ['year', 'land_state_node', 'land_state_meaning', 'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type', 'adm0', 'country_name', 'region_L1', 'region_L2_L3', 'continent', 'ecozone', 'continent_ecozone', 'climate_domain', 'cont_eco', 'Landmark', 'starting_composite_primary_forest', 'KBA', 'watershed', 'watershed_name', 'drivers_of_TCL_1_km', 'driver_1km_text', 'forest_age_category_end_of_interval', 'WDPA', 'WDPA_type', 'WDPA_high_protection', 'first_year_LC_composite_name', 'veg__AGC_emission_factor_CO2_only__fraction', 'veg__carbon_stock__non_soil__MgC', 'veg__gross_emissions__AGC__MgCO2_yr', 'veg__gross_emissions__BGC__MgCO2_yr', 'veg__gross_emissions__CH4__MgCO2e_yr', 'veg__gross_emissions__N2O__MgCO2e_yr', 'veg__gross_emissions__all_C_pools__CO2_only__MgCO2_yr', 'veg__gross_emissions__all_C_pools__all_gases__MgCO2e_yr', 'veg__gross_emissions__all_C_pools__non_CO2_only__MgCO2e_yr', 'veg__gross_emissions__deadwood_C__MgCO2_yr', 'veg__g

In [11]:
# Adds summative LULUCF columns to table
def add_summative_columns(df):

    print(f"  Creating summative columns")
    # print(f"  Creating soil emissions columns")
    df['org_soil_emis_total__MgCO2e_yr']               = df['org_soil_drained_total__MgCO2e_yr'] + df['org_soil_burned_total__MgCO2e_yr']
    df['soil_emis_total__MgCO2e_yr']                   = df['org_soil_emis_total__MgCO2e_yr'] + df['SOC_loss__mineral_soil_extent__0_30cm_MgCO2_yr']

    # print(f"  Creating summative emissions columns")
    df['LULUCF_gross_emissions__CO2__MgCO2_yr']        = df['veg__gross_emissions__all_C_pools__CO2_only__MgCO2_yr'] \
                                                            + df['org_soil_drained_CO2_onsite__MgCO2_yr']  + df['org_soil_drained_CO2_offsite__MgCO2_yr'] + df['org_soil_burned_total_CO2__MgCO2_yr'] \
                                                            + df['SOC_loss__mineral_soil_extent__0_30cm_MgCO2_yr']
    df['LULUCF_gross_emissions__CH4__MgCO2e_yr']       = df['veg__gross_emissions__CH4__MgCO2e_yr'] + df['org_soil_drained_total_CH4__MgCO2e_yr'] + df['org_soil_burned_total_CH4__MgCO2e_yr']
    df['LULUCF_gross_emissions__N2O__MgCO2e_yr']       = df['veg__gross_emissions__N2O__MgCO2e_yr'] + df['org_soil_drained_N2O__MgCO2e_yr']
    df['LULUCF_gross_emissions__non_CO2__MgCO2e_yr']   = df['LULUCF_gross_emissions__CH4__MgCO2e_yr'] + df['LULUCF_gross_emissions__N2O__MgCO2e_yr']
    df['LULUCF_gross_emissions__all_gases__MgCO2e_yr'] = df['LULUCF_gross_emissions__CO2__MgCO2_yr'] + df['LULUCF_gross_emissions__non_CO2__MgCO2e_yr']
    
    # print(f"  Creating summative removals column")
    df['LULUCF_gross_removals__MgCO2_yr']              = df['veg__gross_removals__all_C_pools__MgCO2_yr'] + df['SOC_gain__mineral_soil_extent__0_30cm_MgCO2_yr']

    # print(f"  Creating summative net flux column")
    df['LULUCF_net_flux__MgCO2e_yr']                   = df['LULUCF_gross_emissions__all_gases__MgCO2e_yr'] + df['LULUCF_gross_removals__MgCO2_yr']

    # print(f"  Creating summative carbon stock")
    df['LULUCF_carbon_stock__all_pools_full_extent__MgC']  = df['veg__carbon_stock__non_soil__MgC'] + df['SOC_stock__full_extent__0_30cm_MgC']

    return df

In [ ]:
%%time
# Pandas couldn't handle merging the three component tables (mostly because vegetation is so large). 
# This uses an alternative approach where less is kept in memory by joining the three component tables one year at a time, then merging all years back together.
# If it runs out of memory (kernel crashes) during the processing of each year, just rerun from a clean kernel. It will skip any years that have already been done.
# Takes roughly 8.5 minutes

shared_contextual_cols = veg_contextual_cols
chunks_dir = f'{LULUCF_zonal_stats_folder}/years_global/'
os.makedirs(chunks_dir, exist_ok=True)

# Cast join keys to str in the two small in-memory tables
for col in shared_contextual_cols:
    SOC_df_post_2016_years_filled_in_wide[col]  = SOC_df_post_2016_years_filled_in_wide[col].astype(str)
    org_soil_post_2016_filled_in_wide[col]   = org_soil_post_2016_filled_in_wide[col].astype(str)

years = sorted(pd.read_parquet(veg_parquet_path, columns=['year'])['year'].unique())
print(f"Processing {len(years)} years: {years}")

for year in years:
    out_path = f'{chunks_dir}year_{year}.parquet'
    if os.path.exists(out_path):
        print(f"  Year {year}: already done, skipping")
        continue

    print(f"  Year {year}...")
    veg_year = pd.read_parquet(veg_parquet_path, filters=[('year', '==', year)])
    for col in shared_contextual_cols:
        veg_year[col] = veg_year[col].astype(str)

    soc_year = SOC_df_post_2016_years_filled_in_wide[SOC_df_post_2016_years_filled_in_wide['year'] == str(year)]
    org_year = org_soil_post_2016_filled_in_wide[org_soil_post_2016_filled_in_wide['year'] == str(year)]

    chunk = (
        veg_year
        .merge(soc_year, on=shared_contextual_cols, how='outer')
        .merge(org_year,  on=shared_contextual_cols, how='outer')
    )
    chunk.to_parquet(out_path, index=False)
    print(f"    {len(chunk):,} rows → saved to {out_path}")
    del veg_year, soc_year, org_year, chunk; gc.collect()


### Combine year chunks from disk — stream one year at a time to avoid loading all into memory
print("Combining year chunks...")
year_files = sorted(glob.glob(f'{chunks_dir}year_*.parquet'))
out_path = f"{LULUCF_zonal_stats_folder}/LULUCF__v{cn.LULUCF_model_version_underscore}__LULUCF_summative_vars__wide__{today}.parquet"

# Derive schema and fillna from first year only.
# This reads the first year into the table.
print(f"Adding 2016 to merged table")
first_year = pd.read_parquet(year_files[0])
all_analysis_cols = [c for c in first_year.columns if any(t in c for t in ['MgC', 'fraction', 'area'])]  # Analysis columns derived from first table
print(f"all_analysis_cols before summative columns added: {all_analysis_cols}")
first_year[all_analysis_cols] = first_year[all_analysis_cols].fillna(0)
first_year = add_summative_columns(first_year)
schema = pa.Schema.from_pandas(first_year, preserve_index=False)
all_analysis_cols_summative = [c for c in first_year.columns if any(t in c for t in ['MgC', 'fraction', 'area'])]  # Analysis columns derived from first table
print(f"all_analysis_cols after summative columns added: {all_analysis_cols_summative}")  # To confirm that summative columns have been added

# Reads the rest of the years into the table
with pq.ParquetWriter(out_path, schema) as writer:
    writer.write_table(pa.Table.from_pandas(first_year, schema=schema, preserve_index=False))
    print(f"{year_files[0].split('/')[-1]}: {len(first_year):,} rows")
    del first_year; gc.collect()

    for f in year_files[1:]:
        print(f"Adding {f} to merged table")
        year_table = pd.read_parquet(f)
        year_table[all_analysis_cols] = year_table[all_analysis_cols].fillna(0)
        year_table = add_summative_columns(year_table)

        writer.write_table(pa.Table.from_pandas(year_table, schema=schema, preserve_index=False))
        print(f"  {f.split('/')[-1]}: {len(year_table):,} rows")
        del year_table; gc.collect()

total_rows = pq.read_metadata(out_path).num_rows
print(f"\nRows in LULUCF_wide: {total_rows:,}")
print(f"Saved to {out_path}")

Processing 9 years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
  Year 2016...
    2,739,401 rows → saved to /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/LULUCF_v1_0_0__veg_v1_0_5__minsoil_v1_0_1__orgsoil_v1_0_1/years_global/year_2016.parquet
  Year 2017...
    3,183,287 rows → saved to /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/LULUCF_v1_0_0__veg_v1_0_5__minsoil_v1_0_1__orgsoil_v1_0_1/years_global/year_2017.parquet
  Year 2018...
    3,256,995 rows → saved to /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/LULUCF_v1_0_0__veg_v1_0_5__minsoil_v1_0_1__orgsoil_v1_0_1/years_global/year_2018.parquet
  Year 2019...
    3,329,929 rows → saved to /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/LULUCF_v1_0_0__veg_v1_0_5__minsoil_v1_0_1__orgsoil_v1_0_1/years_global/year_2019.parquet
  Year 2020...
    3,403,582 rows → saved to /mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_stati

In [12]:
# Counts the rows in the three wide component tables and the full LULUCF table. They should match since the three component tables share no contextual layer combinations
indiv_table_row_sum = veg_row_count + SOC_table_shape[0] + org_soil_table_shape[0]
print(f"indiv_table_row_sum: {indiv_table_row_sum}")

LULUCF_wide_metadata = pq.read_metadata(f"{LULUCF_zonal_stats_folder}/LULUCF__v{cn.LULUCF_model_version_underscore}__LULUCF_summative_vars__wide__{today}.parquet")
LULUCF_wide_rows = LULUCF_wide_metadata.num_rows
print(f"LULUCF_wide.shape rows: {LULUCF_wide_rows}")

indiv_table_row_sum: 29375556
LULUCF_wide.shape rows: 29375556


In [14]:
# Reads contextual columns from LULUCF table and prints the options for them

LULUCF_contextual_cols = [c for c in pq.read_schema(
    f"{LULUCF_zonal_stats_folder}/LULUCF__v{cn.LULUCF_model_version_underscore}__LULUCF_summative_vars__wide__{today}.parquet"
).names if not any(t in c for t in ['MgC', 'MgCO2', 'fraction', 'area_ha'])]

LULUCF_wide_contextual = pd.read_parquet(
    f"{LULUCF_zonal_stats_folder}/LULUCF__v{cn.LULUCF_model_version_underscore}__LULUCF_summative_vars__wide__{today}.parquet",
    columns=LULUCF_contextual_cols
)

print(f"Contextual columns are: {LULUCF_contextual_cols}")
for column in LULUCF_contextual_cols:
    print(f"{column}: {np.sort(LULUCF_wide_contextual[column].unique())}")

Contextual columns are: ['year', 'land_state_node', 'land_state_meaning', 'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type', 'adm0', 'country_name', 'region_L1', 'region_L2_L3', 'continent', 'ecozone', 'continent_ecozone', 'climate_domain', 'cont_eco', 'Landmark', 'starting_composite_primary_forest', 'KBA', 'watershed', 'watershed_name', 'drivers_of_TCL_1_km', 'driver_1km_text', 'forest_age_category_end_of_interval', 'WDPA', 'WDPA_type', 'WDPA_high_protection', 'first_year_LC_composite_name']
year: ['2016' '2017' '2018' '2019' '2020' '2021' '2022' '2023' '2024']
land_state_node: ['-999' '11200000' '12100000' '12200000' '12300000' '12400000' '12500000'
 '12600000' '12700000' '13200000' '21100000' '21200000' '22100000'
 '22200000' '31120000' '31190000' '31211200' '31211900' '31212200'
 '31212900' '31221200' '31221900' '31222200' '31222900' '31231200'
 '31231900' '31232200' '31232900' '31241200' '31242200' '32112000'
 '32119000' '32121200' '32121900' '32122200' '32122

In [16]:
# Simplified LULUCF table (wide-format) for using in graphs
# Reads one year at a time (in chunks of ~million rows) because the table is too large to read into memory all at once

parquet_path = f"{LULUCF_zonal_stats_folder}/LULUCF__v{cn.LULUCF_model_version_underscore}__LULUCF_summative_vars__wide__20260617.parquet"

group_cols = ['country_name', 'adm0', 'region_L1', 'region_L2_L3',
              'year',
              'land_state_node', 'land_state_meaning', 'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type',
              'climate_domain']

all_cols      = pq.read_schema(parquet_path).names
analysis_cols = [c for c in all_cols if any(t in c for t in ['MgC', 'fraction', 'area'])]

cols_to_read  = group_cols + analysis_cols
print("Grouping parquet year by year in ~1 million rows at a time ...")
pf = pq.ParquetFile(parquet_path)
print(f"  {pf.metadata.num_row_groups} row groups")

chunks = []
for i in range(pf.metadata.num_row_groups):
    chunk_df = pf.read_row_group(i, columns=cols_to_read).to_pandas()
    grouped  = chunk_df.groupby(group_cols, dropna=False)[analysis_cols].sum().reset_index()
    chunks.append(grouped)
    print(f"  Row group {i+1}: {len(chunk_df):,} rows → {len(grouped):,} grouped rows")
    del chunk_df; gc.collect()

print("Combining grouped chunks...")
LULUCF_for_figures = (
    pd.concat(chunks, ignore_index=True)
    .groupby(group_cols, dropna=False)[analysis_cols]
    .sum()
    .reset_index()
)
del chunks; gc.collect()

print(LULUCF_for_figures.shape)
LULUCF_for_figures.to_parquet(f"{LULUCF_zonal_stats_folder}/LULUCF__v{cn.LULUCF_model_version_underscore}__for_figures__wide__{today}.parquet", index=False)
print("Done summarizing")

Grouping parquet year by year in ~1 million rows at a time ...
  35 row groups
  Row group 1: 1,048,576 rows → 8,797 grouped rows
  Row group 2: 1,048,576 rows → 4,465 grouped rows
  Row group 3: 642,249 rows → 372 grouped rows
  Row group 4: 1,048,576 rows → 8,597 grouped rows
  Row group 5: 1,048,576 rows → 1,993 grouped rows
  Row group 6: 1,048,576 rows → 3,817 grouped rows
  Row group 7: 37,559 rows → 24 grouped rows
  Row group 8: 1,048,576 rows → 8,402 grouped rows
  Row group 9: 1,048,576 rows → 1,595 grouped rows
  Row group 10: 1,048,576 rows → 4,213 grouped rows
  Row group 11: 111,267 rows → 79 grouped rows
  Row group 12: 1,048,576 rows → 8,419 grouped rows
  Row group 13: 1,048,576 rows → 1,582 grouped rows
  Row group 14: 1,048,576 rows → 4,264 grouped rows
  Row group 15: 184,201 rows → 118 grouped rows
  Row group 16: 1,048,576 rows → 8,358 grouped rows
  Row group 17: 1,048,576 rows → 1,567 grouped rows
  Row group 18: 1,048,576 rows → 4,361 grouped rows
  Row group 1

### QC: Checking if values in final LULUCF table match original tables for each component (chunk stats for veg and SOC, input wide table for organic soil)

In [17]:
# Vegetation chunk stats output (for comparison with zonal stats)
veg_chunk_stats_folder = f'/mnt/c/GIS/git/AFOLU_GHG_flux_model/chunk_stats/parquet_20260131_10_37_46__KEEP/'
veg_gross_outputs_parquet = f'{veg_chunk_stats_folder}vegetation_fluxes_20260131_10_37_28__v1_0_5__gross_outputs_1x1.parquet'
veg_net_outputs_parquet = f'{veg_chunk_stats_folder}vegetation_fluxes_20260131_10_37_28__v1_0_5__net_outputs_1x1.parquet'
veg_chunk_stats_gross = pd.read_parquet(veg_gross_outputs_parquet)
veg_chunk_stats_net = pd.read_parquet(veg_net_outputs_parquet)
# Combine gross and net chunk stats, aggregate to 10x10 deg tile level
veg_chunk_stats_combined = pd.concat([veg_chunk_stats_gross, veg_chunk_stats_net], ignore_index=True)

# Vegetation zonal stats before combining with soil
veg_only_file = f"{veg_zonal_stats_folder}/veg_model_zonal_stats__global__v1_0_5__all_layers__wide__20260616.parquet"
veg_zonal_stats = pd.read_parquet(veg_only_file)

In [ ]:
### Compares global annual veg fluxes in final, simplified LULUCF table against original vegetation chunk stats table.
### Uses the simplified LULUCF table rather than the full one because the full one really pushes memory. 
### I expect that chunk stats and zonal stats will differ somewhat for emissions. Emissions chunk stats should be lower because when I ran veg v1.0.5 and got chunk stats, 
### chunk stats summing still didn't sum in chunks with NaN pixels and maybe had other issues.
### So, chunk stats should be lower than zonal stats for emissions in general.
### Also, chunk stats uses the original global run of v1.0.5, which doesn't include the re-run tiles in Canada that included the missing emission factor for partial disturbance,
### so chunk stats also uses slightly older data that is actually missing emissions compared to zonal stats (that integrates the Canada re-run). 

### Chunk stats

# Chunk stats summed across all tiles
print("Summing chunk stats across tiles")
veg_chunk_tile_agg = (
    veg_chunk_stats_combined
    .groupby(['pattern', 'years'], dropna=False)['sum_value']
    .sum()
    .reset_index()
    .rename(columns={'years': 'year', 'pattern': 'variable', 'sum_value': 'tile_sum'})
)
veg_chunk_tile_agg['year'] = veg_chunk_tile_agg['year'].astype(int)
veg_chunk_tile_agg.rename(columns={'variable': 'analysis_layer', 'tile_sum': 'chunk_stats_sum'}, inplace=True)
veg_chunk_tile_agg

# Reference: veg chunk stats -- sum across all tiles by year and analysis_layer
print("Aggregating chunk stats across tiles")
chunk_veg_ref = (
    veg_chunk_tile_agg
    .groupby(['year', 'analysis_layer'])['chunk_stats_sum']
    .sum()
    .unstack('analysis_layer')
    .rename_axis(None, axis='columns')
    .rename(columns=lambda col: col.replace('_ha_yr', ''))
    .sort_index()
)
chunk_veg_ref.index = chunk_veg_ref.index.astype(int)
print("chunk stats")
display(chunk_veg_ref)


### Final LULUCF table (zonal stats)

LULUCF_for_figures  = pd.read_parquet(f"{LULUCF_zonal_stats_folder}/LULUCF__v{cn.LULUCF_model_version_underscore}__for_figures__wide__20260616.parquet")
LULUCF_analysis_cols = [col for col in LULUCF_for_figures.columns 
                        if any(term in col for term in ['MgC', 'fraction', 'area'])]  # Identifies non-contextual columns

# From LULUC simplified for figures -- veg flux columns only, summed globally by year
flux_cols_veg = [col for col in LULUCF_analysis_cols if '__area_ha' not in col]
LULUCF_veg_by_year = (
    LULUCF_for_figures
    .groupby('year')[flux_cols_veg]
    .sum()
    .sort_index()
)
LULUCF_veg_by_year.index = LULUCF_veg_by_year.index.astype(int)
print("final zonal stats")
display(LULUCF_veg_by_year)


### Comparison between chunk stats and LULUCF table.
### The chunk stats values are expected to be lower than the zonal stats-based LULUCF values because when I ran chunk stats originally, it omitted sums for chunks with NaN. So, chunk stats is an under-estimate. 
### Only compare columns that exist in both

# Handles that columns have been renamed in LULUCF table
col_mapping = {col: f'veg__{col}_yr' for col in chunk_veg_ref.columns}

valid = {chunk_col: lulucf_col for chunk_col, lulucf_col in col_mapping.items()
         if chunk_col in chunk_veg_ref.columns and lulucf_col in LULUCF_veg_by_year.columns}
print(f"Comparing {len(valid)} column pairs")

chunk_vals  = chunk_veg_ref[[*valid.keys()]]
lulucf_vals = LULUCF_veg_by_year[[*valid.values()]].rename(columns={v: k for k, v in valid.items()})

diff     = (lulucf_vals - chunk_vals).round(4)
pct_diff = (diff / chunk_vals.abs() * 100).round(8)

print("\n=== Absolute difference (LULUCF minus chunk stats) ===")
display(diff)
print("\n=== Percent difference ===")
display(pct_diff)

In [ ]:
### Compares vegetation-only zonal stats table against final LULUCF table to make sure nothing big changed along the way
### These should be very similar to each other, minus some floating point rounding errors from processing

veg_flux_cols = [c for c in pq.read_schema(veg_only_file).names
                 if c.startswith('veg__') and ('MgCO2' in c or 'MgC' in c or 'fraction' in c)
                 and 'area_ha' not in c]

# ── Sum veg-only table by year ────────────────────────────────────────────────
pf_veg = pq.ParquetFile(veg_only_file)
chunks = []
for i in range(pf_veg.metadata.num_row_groups):
    chunk = pf_veg.read_row_group(i, columns=['year'] + veg_flux_cols).to_pandas()
    chunks.append(chunk.groupby('year')[veg_flux_cols].sum())
    del chunk; gc.collect()
veg_by_year = pd.concat(chunks).groupby('year')[veg_flux_cols].sum()
del chunks; gc.collect()
print("veg-only table summed")


# ── Compare ───────────────────────────────────────────────────────────────────
diff     = (LULUCF_veg_by_year - veg_by_year).round(4)
pct_diff = (diff / veg_by_year.abs() * 100).round(6)

print("\n=== Absolute difference (LULUCF minus veg-only), by year ===")
display(diff)
print("\n=== Percent difference ===")
display(pct_diff)
print("\n=== Global totals match? ===")
display(diff.sum().rename('total_diff'))

In [ ]:
# SOC chunk stats output (for comparison with zonal stats)
SOC_chunk_stats_file = f'/mnt/c/GIS/git/AFOLU_GHG_flux_model/chunk_stats/KEEP_definitive_runs/SOC_density/v1_0_1__2000_2022__20260611/soil_carbon_densities_and_changes_1x1_chunk_statistics_20260611_20_36_25__with_pivot__KEEP.xlsx'
SOC_chunk_stats_gross = pd.read_excel(SOC_chunk_stats_file, sheet_name='other_outputs_1x1')
SOC_chunk_stats_net = pd.read_excel(SOC_chunk_stats_file, sheet_name='net_outputs_1x1')
# Combine gross and net chunk stats, aggregate to 10x10 deg tile level
SOC_chunk_stats_combined = pd.concat([SOC_chunk_stats_gross, SOC_chunk_stats_net], ignore_index=True)

In [ ]:
### Compares global annual SOC fluxes in final, simplified LULUCF table against original SOC chunk stats table.
### Uses the simplified LULUCF table rather than the full one because the full one really pushes memory. 
### I expect that SOC chunk stats and zonal stats should be pretty much identical. 


### Chunk stats

# Chunk stats summed across all tiles
SOC_chunk_tile_agg = (
    SOC_chunk_stats_combined
    .groupby(['pattern', 'years'], dropna=False)['sum_value']
    .sum()
    .reset_index()
    .rename(columns={'years': 'year', 'pattern': 'variable', 'sum_value': 'tile_sum'})
)
SOC_chunk_tile_agg['year'] = SOC_chunk_tile_agg['year'].astype(int)
SOC_chunk_tile_agg.rename(columns={'variable': 'analysis_layer', 'tile_sum': 'chunk_stats_sum'}, inplace=True)

# Final LULUCF table has only post-2016 soil data, so need to limit chunk stats for comparison
SOC_chunk_tile_agg_post_2016 = SOC_chunk_tile_agg[SOC_chunk_tile_agg["year"] == 2020].reset_index(drop=True)
SOC_chunk_tile_agg_post_2016

# Reference: SOC chunk stats -- sum across all tiles by year and analysis_layer
chunk_SOC_ref = (
    SOC_chunk_tile_agg_post_2016
    .groupby(['year', 'analysis_layer'])['chunk_stats_sum']
    .sum()
    .unstack('analysis_layer')
    .rename_axis(None, axis='columns')
    .rename(columns=lambda col: col.replace('_ha_yr', ''))
    .sort_index()
)
chunk_SOC_ref.index = chunk_SOC_ref.index.astype(int)
print("chunk stats")
display(chunk_SOC_ref)


### Final LULUCF table (zonal stats)

LULUCF_for_figures  = pd.read_parquet(f"{LULUCF_zonal_stats_folder}/LULUCF__v{cn.LULUCF_model_version_underscore}__for_figures__wide__20260616.parquet")
LULUCF_analysis_cols = [col for col in LULUCF_for_figures.columns 
                        if any(term in col for term in ['MgC', 'fraction', 'area'])]  # Identifies non-contextual columns

# From LULUCF_for_figures -- veg flux columns only, summed globally by year
flux_cols_SOC = [col for col in LULUCF_analysis_cols if '__area_ha' not in col]
LULUCF_SOC_by_year = (
    LULUCF_for_figures
    .groupby('year')[flux_cols_SOC]
    .sum()
    .sort_index()
)
LULUCF_SOC_by_year.index = LULUCF_SOC_by_year.index.astype(int)

# Limits SOC from LULUCF table to original years
LULUCF_SOC_by_year = LULUCF_SOC_by_year.loc[[2020]]

print("final zonal stats")
display(LULUCF_SOC_by_year)


### Comparison-- should be essentially identical because I fixed chunk stats NaN problem when I reran SOC 
### Only compare columns that exist in both

# Handles that columns have been renamed in LULUCF table
col_mapping = {
    'SOC_density__full_extent__0-30cm_MgC_ha':         'SOC_stock__full_extent__0_30cm_MgC',
    'SOC_density__mineral_soil_extent__0-30cm_MgC_ha': 'SOC_stock__mineral_soil_extent__0_30cm_MgC',
    'SOC_gain__full_extent__0-30cm_MgCO2':             'SOC_gain__full_extent__0_30cm_MgCO2_yr',
    'SOC_gain__mineral_soil_extent__0-30cm_MgCO2':     'SOC_gain__mineral_soil_extent__0_30cm_MgCO2_yr',
    'SOC_loss__full_extent__0-30cm_MgCO2':             'SOC_loss__full_extent__0_30cm_MgCO2_yr',
    'SOC_loss__mineral_soil_extent__0-30cm_MgCO2':     'SOC_loss__mineral_soil_extent__0_30cm_MgCO2_yr',
    'SOC_net__full_extent__0-30cm_MgCO2':              'SOC_net__full_extent__0_30cm_MgCO2_yr',
    'SOC_net__mineral_soil_extent__0-30cm_MgCO2':      'SOC_net__mineral_soil_extent__0_30cm_MgCO2_yr',
}

# Filter to pairs where both sides exist
valid = {chunk_col: lulucf_col for chunk_col, lulucf_col in col_mapping.items()
         if chunk_col in chunk_SOC_ref.columns and lulucf_col in LULUCF_SOC_by_year.columns}
print(f"Comparing {len(valid)} column pairs")

chunk_vals  = chunk_SOC_ref[[*valid.keys()]]
lulucf_vals = LULUCF_SOC_by_year[[*valid.values()]].rename(columns={v: k for k, v in valid.items()})

diff     = (lulucf_vals - chunk_vals).round(4)
pct_diff = (diff / chunk_vals.abs() * 100).round(8)

print("\n=== Absolute difference (LULUCF minus chunk stats) ===")
display(diff)
print("\n=== Percent difference ===")
display(pct_diff)

In [ ]:
### Compares global annual organic soil fluxes in final, simplified LULUCF table against original table.
### Uses the simplified LULUCF table rather than the full one because the full one really pushes memory. 
### I expect that the original and final tables should be identical. 


### Standalone table (after standardization, to simplify comparison)

print("Processing standalone organic soil table")
org_soil_alone  = pd.read_parquet(f"{org_soil_zonal_stats_folder}/org_soil__post_2016__all_vars__wide__20260616.parquet")

# Reference: organic soil table summed by year
flux_cols_org_soil = [col for col in org_soil_alone.columns 
                        if any(term in col for term in ['org_soil_'])] 
org_soil_ref_by_year = (
    org_soil_alone
    .groupby('year')[flux_cols_org_soil]
    .sum()
    .sort_index()
)
org_soil_ref_by_year.index = org_soil_ref_by_year.index.astype(int)
print("standalone organic soil table")
display(org_soil_ref_by_year)


### Final LULUCF table (zonal stats)

print("Processing simplified final LULUCF table")
LULUCF_for_figures  = pd.read_parquet(f"{LULUCF_zonal_stats_folder}/LULUCF__v{cn.LULUCF_model_version_underscore}__for_figures__wide__20260616.parquet")
LULUCF_analysis_cols = [col for col in LULUCF_for_figures.columns 
                        if any(term in col for term in ['MgC', 'fraction', 'area'])]  # Identifies non-contextual columns

# From LULUCF_for_figures -- organic soil flux columns, summed globally by year
LULUCF_org_soil_by_year = (
    LULUCF_for_figures
    .groupby('year')[flux_cols_org_soil]
    .sum()
    .sort_index()
)
LULUCF_org_soil_by_year.index = LULUCF_org_soil_by_year.index.astype(int)


### Comparison-- should be identical

print("final zonal stats")
display(LULUCF_org_soil_by_year)

shared_cols_org_soil = [col for col in flux_cols_org_soil if col in org_soil_ref_by_year.columns]
print(f"Comparing {len(shared_cols_org_soil)} shared organic soil columns")
print(f"Columns in LULUCF_for_figures but not org soil ref: {set(flux_cols_org_soil) - set(org_soil_ref_by_year.columns)}")
print(f"Columns in org soil ref but not LULUCF_for_figures: {set(org_soil_ref_by_year.columns) - set(flux_cols_org_soil)}")

diff = LULUCF_org_soil_by_year[shared_cols_org_soil] - org_soil_ref_by_year[shared_cols_org_soil]
pct_diff = (diff / org_soil_ref_by_year[shared_cols_org_soil].abs() * 100).round(4)

print("\n=== Absolute difference (LULUCF_for_figures minus original org soil) ===")
display(diff)
print("\n=== Percent difference ===")
display(pct_diff)